In [4]:
import numpy as np
import pandas as pd
from pathlib import Path
import subprocess
import optuna

In [ ]:
def read_PES(file: str | Path):
    filename = Path(file)
    pes = np.loadtxt(filename)
    df = pd.DataFrame(
        pes,
        columns=["r12","r13","r23","Eab"]
    )
    return df

def write_inp(
        inp_file: str | Path,
        r12: np.ndarray, r13: np.ndarray, r23: np.ndarray,
        Eab: np.ndarray,
        indx: int, ifor: int, numiter: int, lim: int,
        npnts: int, nordr: int, vex: np.ndarray | list[float], e0: float,
        weights: np.ndarray | None = None
):

    for arr in [r13, r23, Eab]:
        if arr.shape != r12.shape:
            raise ValueError("All arrays must have the same shape.")

    if weights is not None and weights.shape != r12.shape:
        raise ValueError("Weights must have the same shape as the coordinates.")

    with open(inp_file,"w") as f:
        f.write(f"{indx} {ifor} {numiter} {lim} \n")
        f.write(f"{npnts} {nordr}")
        for vxi in vex: 
            f.write(f"{vxi:12.8f} ")
        f.write(f"{e0} \n")

        if weights is None:        
            for i in range(npnts):
                rab = r12[i] ; rac = r13[i] ; rbc = r23[i] ; E = Eab[i]
                f.write(f"{rab:7.3f} {rbc:7.3f} {rac:7.3f} {E:12.8f} \n")
        else:
            for i in range(npnts):
                rab = r12[i] ; rac = r13[i] ; rbc = r23[i] ; E = Eab[i] ; w = weights[i]
                f.write(f"{rab:7.3f} {rbc:7.3f} {rac:7.3f} {E:12.8f} {w:12.8f} \n")

def run_fit(
        exec: str | Path, inp_file: str | Path, workdir: str | Path | None = None,
        timeout: float | None = None
) -> subprocess.CompletedProcess[str]:

    executable = Path(exec).resolve()
    inp_file = Path(inp_file).resolve()
    if workdir is None:
        workdir = inp_file.parent
    workdir = Path(workdir).resolve()

    if not executable.exists():
        raise FileNotFoundError(f"Executable not found: {executable}")
    if not inp_file.exists():
        raise FileNotFoundError(f"Input file not found: {inp_file}")

    with open(inp_file,"r") as f:
        result = subprocess.run(
            [str(executable)],stdin=f,cwd=workdir,
            capture_output=True,text=True,timeout=timeout,check=False
        )

    (workdir / "stdout.log").write_text(result.stdout)
    (workdir / "stderr.log").write_text(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            f"El ajuste ha fallado con código {result.returncode}.\n"
            f"Revisa:\n"
            f"  {workdir / 'stdout.log'}\n"
            f"  {workdir / 'stderr.log'}"
        )

    return result

def rms(values) -> float:
    values = np.asarray(values, dtype=float)

    if values.size == 0:
        raise ValueError("No hay puntos para calcular el RMS.")

    return float(np.sqrt(np.mean(values**2)))

def read_output(file):
    with open(file) as f:
        lines = f.readlines()

    return {
        "iterations": parse_iterations(lines),
        "residuals": parse_residuals(lines),
        "summary": parse_summary(lines),
        "vex": parse_vex(lines)
    }

def parse_iterations(lines: list[str]):
    rows = [] ; reading = False

    for line in lines: 
        if ("iteration" in  line and "rms(u.a.)" in line and "Emax(kcal/mol)" in line):
            reading = True
            continue
        if not reading:
            continue
        if "v-inp" in line and "v-fit" in line:
            break

        values = line.split()

        if len(values) != 6:
            continue

        try: 
            iteration = int(values[0])
            numeric_values = [
                float(value.replace("D","E").replace("d","e"))
                for value in values[1:]
            ]
        except ValueError:
            continue

        rows.append([iteration,*numeric_values])

    if not rows:
        raise ValueError("No iterations found in output")

    return pd.DataFrame(
        rows,
        columns=[
            "iteration","vex1","vex2",
            "rms_au","rms_kcal","emax_kcal"
        ]
    )

def parse_residuals(lines: list[str]):
    rows = [] ; reading = False

    for line in lines:
        if ("v-inp" in line and "v-fit" in line and "diff(u.a.)" in line):
            reading = True
            continue
        if not reading:
            continue
        if "n effectif=" in line: 
            break

        values = line.split()

        if len(values) != 7:
            continue

        try:
            numeric_values = [
                float(value.replace("D", "E").replace("d", "e"))
                for value in values
            ]
        except ValueError:
            continue

        rows.append(numeric_values)

    if not rows:
        raise ValueError("Not residuals found")

    return pd.DataFrame(
        rows,
        columns=[
            "r12","r13","r23",
            "v_inp","v_fit",
            "diff_au","diff_kcal"
        ]
    )

def parse_summary(lines: list[str]) -> dict[str, float]:
    summary = {}

    for line in lines:
        if line.strip().startswith("n effectif="):
            values = line.split()
            summary["neff"] = int(values[-1])
        if line.strip().startswith("RMS="):
            values = line.replace("D", "E").split()

            summary["rms_au"] = float(values[-5])
            summary["rms_kcal"] = float(values[-2])

        elif line.strip().startswith("Emax"):
            values = line.replace("D", "E").split()

            summary["emax_kcal"] = float(values[2])

    required = {"neff","rms_au", "rms_kcal", "emax_kcal"}

    if summary.keys() < required:
        raise ValueError("No se pudo leer el summary completo.")

    return summary

def parse_vex(lines: list[str]) -> list[float]:
    for line in reversed(lines):
        if line.strip().startswith("vex1(1)="):
            values = line.replace("D", "E").replace("d", "e").split()

            try:
                return [
                    float(values[1]),
                    float(values[-1]),
                ]
            except (ValueError, IndexError) as exc:
                raise ValueError(
                    f"No se pudieron leer los parámetros finales: {line.strip()}"
                ) from exc

    raise ValueError("No se encontraron los parámetros vex finales.")

def sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


def generate_sigmoid_weights(
    pes: pd.DataFrame,
    r_h2_eq: float,r_ph_eq: float,
    amp_entrance: float = 10.0,amp_exit: float = 10.0,
    r_entrance: float = 8.0,r_exit: float = 8.0,
    sigma_h2: float = 0.20,sigma_ph: float = 0.30,
    sharpness: float = 2.0,base_weight: float = 1.0,
) -> np.ndarray:
    required_columns = {"r12", "r13", "r23"}

    if not required_columns.issubset(pes.columns):
        missing = required_columns - set(pes.columns)
        raise KeyError(f"Faltan columnas en el DataFrame: {sorted(missing)}")

    r12 = pes["r12"].to_numpy(dtype=float)
    r13 = pes["r13"].to_numpy(dtype=float)
    r23 = pes["r23"].to_numpy(dtype=float)

    p_far = sigmoid(sharpness * (np.minimum(r12, r13) - r_entrance))

    h2_character = np.exp(
        -0.5 * ((r23 - r_h2_eq) / sigma_h2) ** 2
    )

    entrance_score = p_far * h2_character

    exit_1 = (
        sigmoid(sharpness * (r13 - r_exit))
        * np.exp(-0.5 * ((r12 - r_ph_eq) / sigma_ph) ** 2)
    )
    exit_2 = (
        sigmoid(sharpness * (r12 - r_exit))
        * np.exp(-0.5 * ((r13 - r_ph_eq) / sigma_ph) ** 2)
    )

    exit_score = np.maximum(exit_1, exit_2)

    weights = (
        base_weight
        + amp_entrance * entrance_score
        + amp_exit * exit_score
    )

    return weights

In [6]:
def generate_channel_scores(
    pes,
    r_h2_eq: float,r_ph_eq: float,
    r_entrance: float,r_exit: float,
    sigma_h2: float,sigma_ph: float,
    sharpness: float,
):
    r12 = pes["r12"].to_numpy(dtype=float)
    r13 = pes["r13"].to_numpy(dtype=float)
    r23 = pes["r23"].to_numpy(dtype=float)

    def sigmoid(x):
        return 1.0 / (1.0 + np.exp(-x))

    entrance_score = (
        sigmoid(sharpness * (np.minimum(r12, r13) - r_entrance))
        * np.exp(-0.5 * ((r23 - r_h2_eq) / sigma_h2) ** 2)
    )

    exit_1_score = (
        sigmoid(sharpness * (r13 - r_exit))
        * np.exp(-0.5 * ((r12 - r_ph_eq) / sigma_ph) ** 2)
    )

    exit_2_score = (
        sigmoid(sharpness * (r12 - r_exit))
        * np.exp(-0.5 * ((r13 - r_ph_eq) / sigma_ph) ** 2)
    )

    exit_score = np.maximum(exit_1_score, exit_2_score)

    return entrance_score, exit_score

In [17]:
def objective(
    trial: optuna.Trial,
    pes,executable: str | Path,runs_dir: str | Path,vex,*,
    indx_i2: int,ifor_i2: int,numiter: int,lim: float,nordr: int,e0: float,
    r_h2_eq: float,r_ph_eq: float,r_entrance: float,r_exit: float,
    sigma_h2: float,sigma_ph: float,sharpness: float,
) -> float:

    runs_dir = Path(runs_dir)
    run_dir = runs_dir / f"trial_{trial.number:05d}"
    run_dir.mkdir(parents=True, exist_ok=False)

    # Parámetros que optimiza Optuna
    amp_entrance = trial.suggest_float(
        "amp_entrance",0.1,100.0,log=True,
    )

    amp_exit = trial.suggest_float(
        "amp_exit",0.1,100.0,log=True,
    )

    # Construcción de pesos para este trial
    weights = generate_sigmoid_weights(
        pes=pes,
        r_h2_eq=r_h2_eq,r_ph_eq=r_ph_eq,
        amp_entrance=amp_entrance,amp_exit=amp_exit,
        r_entrance=r_entrance,r_exit=r_exit,
        sigma_h2=sigma_h2,sigma_ph=sigma_ph,
        sharpness=sharpness,base_weight=1.0,
    )

    inp_file = run_dir / "fit.inp"
    output_file = run_dir / "out.fit"

    # Escribir input
    write_inp(
        inp_file=inp_file,
        r12=pes["r12"].to_numpy(copy=True),r13=pes["r13"].to_numpy(copy=True),r23=pes["r23"].to_numpy(copy=True),Eab=pes["Eab"].to_numpy(copy=True),weights=weights,
        indx=indx_i2,ifor=ifor_i2,numiter=numiter,lim=lim,npnts=len(pes),nordr=nordr,vex=vex,e0=e0,
    )

    # Ejecutar Fortran
    result = run_fit(
        exec=executable,inp_file=inp_file,workdir=run_dir
    )

    output_file.write_text(result.stdout)

    # Leer output
    output = read_output(output_file)

    residuals = output["residuals"]
    summary = output["summary"]

    if len(residuals) != len(pes):
        raise ValueError(
            "El número de residuos no coincide con el número de puntos: "
            f"{len(residuals)} != {len(pes)}"
        )

    entrance_score, exit_score = generate_channel_scores(
        pes=pes,
        r_h2_eq=r_h2_eq,r_ph_eq=r_ph_eq,
        r_entrance=r_entrance,r_exit=r_exit,
        sigma_h2=sigma_h2,sigma_ph=sigma_ph,
        sharpness=sharpness,
    )

    channel_threshold = 0.5

    entrance_mask = entrance_score >= channel_threshold
    exit_mask = exit_score >= channel_threshold
    interaction_mask = ~(entrance_mask | exit_mask)

    errors = residuals["diff_kcal"].to_numpy(dtype=float)

    rms_global = rms(errors)

    rms_entrance = (
        rms(errors[entrance_mask])
        if np.any(entrance_mask)
        else rms_global
    )

    rms_exit = (
        rms(errors[exit_mask])
        if np.any(exit_mask)
        else rms_global
    )

    rms_interaction = (
        rms(errors[interaction_mask])
        if np.any(interaction_mask)
        else rms_global
    )

    emax = float(np.max(np.abs(errors)))

    # Guardar información útil dentro del trial
    trial.set_user_attr("rms_global", rms_global)
    trial.set_user_attr("rms_entrance", rms_entrance)
    trial.set_user_attr("rms_exit", rms_exit)
    trial.set_user_attr("rms_interaction", rms_interaction)
    trial.set_user_attr("emax_kcal", emax)

    trial.set_user_attr(
        "n_entrance",
        int(np.count_nonzero(entrance_mask)),
    )
    trial.set_user_attr(
        "n_exit",
        int(np.count_nonzero(exit_mask)),
    )
    trial.set_user_attr(
        "n_interaction",
        int(np.count_nonzero(interaction_mask)),
    )

    if "vex" in output:
        trial.set_user_attr(
            "final_vex",
            np.asarray(output["vex"]).tolist(),
        )

    # Función objetivo externa
    score = (
        0.15 * rms_global
        + 0.35 * rms_entrance
        + 0.35 * rms_exit
        + 0.10 * rms_interaction
        + 0.05 * emax
    )

    return float(score)

In [18]:
pes = read_PES("/home/jorgebdelafuente/Doctorado/Fit_PH2M/1SAp/1SAp/1SAp.dat")
execut = "/home/jorgebdelafuente/Doctorado/Fit_PH2M/1SAp/a.out"

In [19]:
study = optuna.create_study(
    study_name="channel_weight_optimization",storage="sqlite:///channel_weights.db",load_if_exists=True,direction="minimize",
)

study.optimize(
    lambda trial: objective(
        trial=trial,pes=pes,
        executable=execut,runs_dir="./runs",
        vex=np.array([0.8, 0.5]),indx_i2=1,ifor_i2=1,numiter=300,lim=0,nordr=6,e0=0.0,
        r_h2_eq=1.40,r_ph_eq=2.70,
        r_entrance=8.0,r_exit=8.0,
        sigma_h2=0.20,sigma_ph=0.30,
        sharpness=2.0,
    ),
    n_trials=100,
    gc_after_trial=True,
)

[I 2026-08-03 14:36:12,289] Using an existing study with name 'channel_weight_optimization' instead of creating a new one.
[W 2026-08-03 14:36:12,439] Trial 4 failed with parameters: {'amp_entrance': 8.155994781738604, 'amp_exit': 0.609508201059928} because of the following error: ValueError('No iterations found in output').
Traceback (most recent call last):
  File "/home/jorgebdelafuente/miniconda3/envs/gnrl_env/lib/python3.13/site-packages/optuna/study/_optimize.py", line 205, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_65205/722258977.py", line 6, in <lambda>
    lambda trial: objective(
                  ~~~~~~~~~^
        trial=trial,pes=pes,
        ^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
        sharpness=2.0,
        ^^^^^^^^^^^^^^
    ),
    ^
  File "/tmp/ipykernel_65205/4275415412.py", line 50, in objective
    output = read_output(output_file)
  File "/tmp/ipykernel_65205/600363400.py", line 90, in read_output
    "iterations": parse_iterations(

ValueError: No iterations found in output

In [10]:
entrance_score, exit_score = generate_channel_scores(
    pes=pes,
    r_h2_eq=1.40,
    r_ph_eq=2.70,
    r_entrance=8.0,
    r_exit=8.0,
    sigma_h2=0.20,
    sigma_ph=0.30,
    sharpness=2.0,
)

print("Entrada:", np.count_nonzero(entrance_score >= 0.5))
print("Salida:", np.count_nonzero(exit_score >= 0.5))
print(
    "Interacción:",
    np.count_nonzero(
        ~(
            (entrance_score >= 0.5)
            | (exit_score >= 0.5)
        )
    ),
)

print("Entrada score max:", entrance_score.max())
print("Salida score max:", exit_score.max())

Entrada: 132
Salida: 1272
Interacción: 28596
Entrada score max: 0.9999999999631621
Salida score max: 0.9999999997647018
